In [0]:
from pyspark.sql.functions import col, when, lpad, concat, lit, to_timestamp, substring, to_date, coalesce, year, month, dayofmonth, dayofweek, hour, minute, to_utc_timestamp

print("1. Reading 28.9 Million Rows from Bronze Layer...")
bronze_flights = spark.table("aviation_project.bronze_historical_flights")

In [0]:
# ---------------------------------------------------------
# Part 1: The LIT Filter
# ---------------------------------------------------------
print("2. Filtering for LIT flights...")
lit_flights = bronze_flights.filter((col("ORIGIN") == "LIT") | (col("DEST") == "LIT"))

In [0]:
# ---------------------------------------------------------
# Part 2: Defensive Timestamp Reconstruction
# ---------------------------------------------------------
print("3. Reconstructing Local Timestamps...")
clean_date = coalesce(
    to_date(col("FL_DATE"), "M/d/yyyy h:mm:ss a"),
    to_date(col("FL_DATE"), "M/d/yyyy"),
    to_date(col("FL_DATE"), "yyyy-MM-dd") 
)
date_str = clean_date.cast("string")

padded_time = lpad(col("CRS_DEP_TIME").cast("string"), 4, "0")
hour_str = substring(padded_time, 1, 2)
minute_str = substring(padded_time, 3, 2)

timestamp_str = concat(date_str, lit(" "), hour_str, lit(":"), minute_str, lit(":00"))

# Create the LOCAL timestamp temporarily
df_with_local_ts = lit_flights.withColumn("local_scheduled_time", to_timestamp(timestamp_str, "yyyy-MM-dd HH:mm:ss"))
df_with_local_ts = df_with_local_ts.dropna(subset=["local_scheduled_time"])

In [0]:
# ---------------------------------------------------------
# Part 3: The Silver Data Contract & Leakage Guard
# ---------------------------------------------------------
print("4. Extracting ML Features & Shifting to UTC...")

flight_timezone_expr = when(col("ORIGIN").isin("ATL", "CLT", "LGA", "DCA", "MIA"), "America/New_York") \
                .when(col("ORIGIN").isin("LIT", "DFW", "ORD", "IAH", "DAL", "STL"), "America/Chicago") \
                .when(col("ORIGIN") == "DEN", "America/Denver") \
                .when(col("ORIGIN") == "LAS", "America/Los_Angeles") \
                .otherwise("UTC")

silver_flights = df_with_local_ts.select(
    # 1. Chronological Anchor (UTC for joining weather)
    to_utc_timestamp(col("local_scheduled_time"), flight_timezone_expr).alias("scheduled_time_utc"),
    
    # 2. Temporal Features (Strictly from LOCAL time so model understands rush hour)
    year(col("local_scheduled_time")).alias("year"),
    month(col("local_scheduled_time")).alias("month"),
    dayofmonth(col("local_scheduled_time")).alias("day_of_month"),
    dayofweek(col("local_scheduled_time")).alias("day_of_week"),
    hour(col("local_scheduled_time")).alias("hour"),
    minute(col("local_scheduled_time")).alias("minute"),
    
    # 3. Flight Identifiers (String to prevent categorical dropping)
    col("OP_UNIQUE_CARRIER").alias("airline_code"),
    col("OP_CARRIER_FL_NUM").cast("int").cast("string").alias("flight_number"),
    
    # 4. Routing Airports
    col("ORIGIN").alias("origin_airport"),
    col("DEST").alias("destination_airport"),
    
    # 5. Machine Learning Targets
    col("DEP_DELAY").cast("int").alias("delay_minutes"),
    when(col("DEP_DELAY") > 15, 1).otherwise(0).alias("is_delayed")
)


In [0]:
display(silver_flights.orderBy("scheduled_time_utc").limit(10))

In [0]:
# ---------------------------------------------------------
# Part 4: Write and Optimize
# ---------------------------------------------------------
print("5. Writing to Silver Delta Table...")
silver_flights.write.format("delta").mode("overwrite").saveAsTable("aviation_project.silver_historical_flights")

In [0]:
spark.sql("OPTIMIZE aviation_project.silver_historical_flights ZORDER BY (scheduled_time_utc)")
print("6. SUCCESS! Historical flights locked in UTC with Local features preserved.")